# Week 8 - Data Cleaning Pipeline

This notebook loads, cleans, and validates the raw e-commerce datasets generated by `generate_data.py`. The final cleaned datasets are exported to the `data/cleaned/` directory.

In [1]:
import pandas as pd
import numpy as np
import os

# Define paths
RAW_DIR = "../data/raw"
CLEANED_DIR = "../data/cleaned"
REPORT_PATH = "../reports"

os.makedirs(CLEANED_DIR, exist_ok=True)
os.makedirs(REPORT_PATH, exist_ok=True)
print("Setup complete. Pandas version:", pd.__version__)

Setup complete. Pandas version: 3.0.3


## 1. Load the Datasets

In [2]:
customers = pd.read_csv(os.path.join(RAW_DIR, "customers.csv"))
products = pd.read_csv(os.path.join(RAW_DIR, "products.csv"))
orders = pd.read_csv(os.path.join(RAW_DIR, "orders.csv"))
order_items = pd.read_csv(os.path.join(RAW_DIR, "order_items.csv"))

print(f"Customers shape: {customers.shape}")
print(f"Products shape: {products.shape}")
print(f"Orders shape: {orders.shape}")
print(f"Order Items shape: {order_items.shape}")

Customers shape: (500, 5)
Products shape: (500, 5)
Orders shape: (700, 5)
Order Items shape: (1200, 6)


## 2. Data Cleaning & Validation

### 2.1 Customers Dataset
* Remove invalid emails (missing `@`).
* Drop any duplicate customer records.

In [3]:
print("Customers Initial Info:")
print(customers.info())

issues = []

# 1. Filter out invalid emails (must contain '@')
invalid_email_mask = ~customers["email"].str.contains("@", na=False)
print(f"\nFound {invalid_email_mask.sum()} invalid emails.")

issues.append(f"Wrong Email formats fixed : {invalid_email_mask.sum()}")

customers_cleaned = customers[~invalid_email_mask].copy()

# 2. Remove duplicate records
duplicate_count = customers_cleaned.duplicated().sum()
print(f"Found {duplicate_count} duplicate rows in customers.")

customers_cleaned = customers_cleaned.drop_duplicates()
print(f"Cleaned Customers shape: {customers_cleaned.shape}")

Customers Initial Info:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   customer_id        500 non-null    int64
 1   customer_name      500 non-null    str  
 2   email              500 non-null    str  
 3   registration_date  500 non-null    str  
 4   customer_type      500 non-null    str  
dtypes: int64(1), str(4)
memory usage: 19.7 KB
None

Found 5 invalid emails.
Found 0 duplicate rows in customers.
Cleaned Customers shape: (495, 5)


### 2.2 Products Dataset
* Clean messy product names (trim leading/trailing whitespaces).

In [4]:
print("Products Initial Info:")
print(products.info())

products_cleaned = products.copy()
messy = (
    products_cleaned["product_name"] !=
    products_cleaned["product_name"].str.strip().str.title()
).sum()

issues.append(f"Messy product names : {messy}")
products_cleaned["product_name"] = products_cleaned["product_name"].str.strip()

print("\nSample product names after trimming whitespaces:")
print(products_cleaned["product_name"].head(10))
print(f"Cleaned Products shape: {products_cleaned.shape}")

Products Initial Info:
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   product_id    500 non-null    int64  
 1   product_name  500 non-null    str    
 2   category      500 non-null    str    
 3   subcategory   500 non-null    str    
 4   cost_price    500 non-null    float64
dtypes: float64(1), int64(1), str(3)
memory usage: 19.7 KB
None

Sample product names after trimming whitespaces:
0    Education
1       Camera
2    Education
3    BIOGRAPHY
4    Furniture
5       Mobile
6       Mobile
7      Kitchen
8    Biography
9          Men
Name: product_name, dtype: str
Cleaned Products shape: (500, 5)


In [5]:
duplicate_count = products_cleaned.duplicated().sum()
print(f"Duplicate count in products: {duplicate_count}")

Duplicate count in products: 0


### 2.3 Orders Dataset
* Handle missing/null `customer_id` values.
* Standardize inconsistent `order_date` formats to a uniform datetime format.

In [6]:
print("Orders Initial Info:")
print(orders.info())

# 1. Drop orders with missing customer_id
null_cust_count = orders["customer_id"].isnull().sum()
print(f"\nOrders with missing customer_id: {null_cust_count}")
orders_cleaned = orders.dropna(subset=["customer_id"]).copy()
orders_cleaned["customer_id"] = orders_cleaned["customer_id"].astype(int)

# 2. Parse dates with mixed formats and standardize
orders_cleaned["order_date"] = pd.to_datetime(orders_cleaned["order_date"], format="mixed")

print(f"Cleaned Orders shape: {orders_cleaned.shape}")

Orders Initial Info:
<class 'pandas.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   order_id     700 non-null    int64  
 1   customer_id  666 non-null    float64
 2   order_date   700 non-null    str    
 3   status       700 non-null    str    
 4   region_code  700 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 27.5 KB
None

Orders with missing customer_id: 34
Cleaned Orders shape: (666, 5)


### 2.4 Order Items Dataset
* Fix negative quantities (convert them to positive values).

In [7]:
print("Order Items Initial Info:")
print(order_items.info())

# Fix negative quantities
negative_qty_count = (order_items["quantity"] < 0).sum()
print(f"\nFound {negative_qty_count} negative quantities.")

order_items_cleaned = order_items.copy()
order_items_cleaned["quantity"] = order_items_cleaned["quantity"].abs()

print(f"Cleaned Order Items shape: {order_items_cleaned.shape}")

Order Items Initial Info:
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   item_id           1200 non-null   int64  
 1   order_id          1200 non-null   int64  
 2   product_id        1200 non-null   int64  
 3   quantity          1200 non-null   int64  
 4   unit_price        1200 non-null   float64
 5   discount_percent  1200 non-null   int64  
dtypes: float64(1), int64(5)
memory usage: 56.4 KB
None

Found 46 negative quantities.
Cleaned Order Items shape: (1200, 6)


## 3. Referential Integrity Validation

* Check that all `customer_id` values in `orders` exist in the `customers` table.
* Check that all `order_id` values in `order_items` exist in the `orders` table.
* Check that all `product_id` values in `order_items` exist in the `products` table.

In [8]:
# Validate Orders -> Customers
valid_customers_mask = orders_cleaned["customer_id"].isin(customers_cleaned["customer_id"])
print(f"Orders with valid customers: {valid_customers_mask.sum()} out of {len(orders_cleaned)}")
orders_cleaned = orders_cleaned[valid_customers_mask]

# Validate Order Items -> Orders
valid_orders_mask = order_items_cleaned["order_id"].isin(orders_cleaned["order_id"])
print(f"Order items with valid orders: {valid_orders_mask.sum()} out of {len(order_items_cleaned)}")
order_items_cleaned = order_items_cleaned[valid_orders_mask]

# Validate Order Items -> Products
valid_products_mask = order_items_cleaned["product_id"].isin(products_cleaned["product_id"])
print(f"Order items with valid products: {valid_products_mask.sum()} out of {len(order_items_cleaned)}")
order_items_cleaned = order_items_cleaned[valid_products_mask]

Orders with valid customers: 659 out of 666
Order items with valid orders: 1136 out of 1200
Order items with valid products: 1136 out of 1136


In [9]:
invalid = order_items[
    ~order_items["order_id"].isin(
        orders_cleaned["order_id"]
    )
]

issues.append(
    f"Invalid order references : {len(invalid)}"
)

## 4. Save Cleaned Datasets

In [10]:
customers_cleaned.to_csv(os.path.join(CLEANED_DIR, "customers_clean.csv"), index=False)
products_cleaned.to_csv(os.path.join(CLEANED_DIR, "products_clean.csv"), index=False)
orders_cleaned.to_csv(os.path.join(CLEANED_DIR, "orders_clean.csv"), index=False)
order_items_cleaned.to_csv(os.path.join(CLEANED_DIR, "order_items_clean.csv"), index=False)

print("All cleaned datasets exported successfully to:", CLEANED_DIR)

All cleaned datasets exported successfully to: ../data/cleaned


In [11]:
with open(
    f"{REPORT_PATH}/issues_report.txt",
    "w"
) as f:

    for issue in issues:
        f.write(issue + "\n")

print("Issues report generated.")

Issues report generated.
